# Kaggle SBD Keyframe Extraction - TransNetV2 + Scene Uniform Sampling

Notebook n?y d?ng ?? chia 15 folder video cho nhi?u account Kaggle ch?y song song.

Output m?i account g?m:

- `Keyframes/<SPLIT_NAME>/<video_id>/keyframe_0000.webp`
- `map-keyframes/<video_id>.csv`
- `metadata_<PART_NAME>.json`
- `video_summary_<PART_NAME>.csv`
- `<PART_NAME>_sbd_keyframes.zip`

C?u h?nh m?c ??nh ?ang d?ng h??ng ?? th?ng nh?t:

```text
--sampling-mode scene-uniform
--sample-interval-seconds 2
--max-frames-per-scene 8
```

Notebook c?ng c? ch? ?? t?n d?ng Kaggle x2 T4: b?t `USE_DUAL_GPU_WORKERS=True` ?? chia danh s?ch video th?nh 2 worker process, m?i process bind v?o m?t GPU b?ng `CUDA_VISIBLE_DEVICES`.

Sau khi t?i c?c zip t? 5 account v? m?y ch?nh, merge keyframes/map/metadata r?i m?i encode embedding v? rebuild index.

In [ ]:
# 1. Install / clone TransNetV2 without relying on Git LFS smudge
from pathlib import Path
import os
import sys
import subprocess
import shutil
import urllib.request

WORKING_DIR = Path('/kaggle/working')
TRANSNET_DIR = WORKING_DIR / 'TransNetV2'
INFERENCE_DIR = TRANSNET_DIR / 'inference'
WEIGHTS_DIR = INFERENCE_DIR / 'transnetv2-weights'

# N?u b?n ?? add Kaggle Model/Dataset ch?a weights, c? th? ?? None ?? auto-discover.
# V?i screenshot hi?n t?i, files c? th? n?m d?ng flat trong m?t folder:
#   saved_model.pb
#   variables.data-00000-of-00001
#   variables.index
# N?u auto-discover kh?ng t?m th?y, set th?ng folder ??, v? d?:
# TRANSNET_WEIGHTS_INPUT = Path('/kaggle/input/transnetv2-weights/transnetv2-weights/2')
TRANSNET_WEIGHTS_INPUT = None

FLAT_WEIGHT_FILES = [
    'saved_model.pb',
    'variables.data-00000-of-00001',
    'variables.index',
]

CANONICAL_WEIGHT_FILES = [
    'saved_model.pb',
    'variables/variables.data-00000-of-00001',
    'variables/variables.index',
]


def run(cmd, **kwargs):
    print('+', ' '.join(map(str, cmd)))
    return subprocess.run(cmd, check=True, **kwargs)


def is_real_weight_file(path: Path, min_bytes: int = 1024 * 1024) -> bool:
    return path.exists() and path.stat().st_size >= min_bytes


def canonical_weights_ready() -> bool:
    return (
        all((WEIGHTS_DIR / rel).exists() for rel in CANONICAL_WEIGHT_FILES)
        and is_real_weight_file(WEIGHTS_DIR / 'saved_model.pb')
    )


def has_canonical_layout(candidate: Path) -> bool:
    return all((candidate / rel).exists() for rel in CANONICAL_WEIGHT_FILES)


def has_flat_layout(candidate: Path) -> bool:
    return all((candidate / rel).exists() for rel in FLAT_WEIGHT_FILES)


def find_kaggle_weights_dir():
    if TRANSNET_WEIGHTS_INPUT is not None:
        candidate = Path(TRANSNET_WEIGHTS_INPUT)
        if has_canonical_layout(candidate) or has_flat_layout(candidate):
            return candidate
        raise FileNotFoundError(f'TRANSNET_WEIGHTS_INPUT exists but does not contain expected weight files: {candidate}')

    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return None

    for saved_model in input_root.rglob('saved_model.pb'):
        candidate = saved_model.parent
        if has_canonical_layout(candidate) or has_flat_layout(candidate):
            return candidate
    return None


def copy_weights_from_source(source_dir: Path):
    print('Copying TransNetV2 weights from:', source_dir)
    WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
    (WEIGHTS_DIR / 'variables').mkdir(parents=True, exist_ok=True)

    if has_canonical_layout(source_dir):
        mapping = {
            source_dir / 'saved_model.pb': WEIGHTS_DIR / 'saved_model.pb',
            source_dir / 'variables' / 'variables.data-00000-of-00001': WEIGHTS_DIR / 'variables' / 'variables.data-00000-of-00001',
            source_dir / 'variables' / 'variables.index': WEIGHTS_DIR / 'variables' / 'variables.index',
        }
    elif has_flat_layout(source_dir):
        mapping = {
            source_dir / 'saved_model.pb': WEIGHTS_DIR / 'saved_model.pb',
            source_dir / 'variables.data-00000-of-00001': WEIGHTS_DIR / 'variables' / 'variables.data-00000-of-00001',
            source_dir / 'variables.index': WEIGHTS_DIR / 'variables' / 'variables.index',
        }
    else:
        return False

    for source, dest in mapping.items():
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, dest)
        print('  copied', source.name, '->', dest)

    return canonical_weights_ready()


def copy_weights_from_kaggle_input():
    source_dir = find_kaggle_weights_dir()
    if source_dir is None:
        return False
    return copy_weights_from_source(source_dir)


def download_weights_from_media_github():
    base_url = 'https://media.githubusercontent.com/media/soCzech/TransNetV2/master/inference/transnetv2-weights'
    WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
    (WEIGHTS_DIR / 'variables').mkdir(parents=True, exist_ok=True)

    mapping = {
        'saved_model.pb': WEIGHTS_DIR / 'saved_model.pb',
        'variables/variables.data-00000-of-00001': WEIGHTS_DIR / 'variables' / 'variables.data-00000-of-00001',
        'variables/variables.index': WEIGHTS_DIR / 'variables' / 'variables.index',
    }
    for rel, dest in mapping.items():
        url = f'{base_url}/{rel}'
        dest.parent.mkdir(parents=True, exist_ok=True)
        print('Downloading', url, '->', dest)
        urllib.request.urlretrieve(url, dest)

    return canonical_weights_ready()


# N?u clone tr??c ?? fail v? Git LFS smudge, th? m?c c? th? t?n t?i nh?ng thi?u inference code.
if TRANSNET_DIR.exists() and not (INFERENCE_DIR / 'transnetv2.py').exists():
    print('Removing incomplete TransNetV2 checkout:', TRANSNET_DIR)
    shutil.rmtree(TRANSNET_DIR)

if not TRANSNET_DIR.exists():
    env = os.environ.copy()
    env['GIT_LFS_SKIP_SMUDGE'] = '1'
    run(['git', 'clone', '--depth', '1', 'https://github.com/soCzech/TransNetV2.git', str(TRANSNET_DIR)], env=env)
else:
    print(f'TransNetV2 already exists: {TRANSNET_DIR}')

if not canonical_weights_ready():
    if not copy_weights_from_kaggle_input():
        try:
            ok = download_weights_from_media_github()
        except Exception as exc:
            ok = False
            print('Media GitHub weight download failed:', repr(exc))
        if not ok:
            raise RuntimeError(
                'TransNetV2 weights are missing. Add a Kaggle Model/Dataset containing either:\n'
                '1) flat files: saved_model.pb, variables.data-00000-of-00001, variables.index; or\n'
                '2) canonical folder: saved_model.pb and variables/variables.*.\n'
                'Then set TRANSNET_WEIGHTS_INPUT to that folder if auto-discover cannot find it.'
            )

sys.path.append(str(INFERENCE_DIR))
print('TransNetV2 path ready:', INFERENCE_DIR)
print('Weights ready:', WEIGHTS_DIR)
print('saved_model.pb size MB:', round((WEIGHTS_DIR / 'saved_model.pb').stat().st_size / (1024 * 1024), 2))
print('variables.data size MB:', round((WEIGHTS_DIR / 'variables' / 'variables.data-00000-of-00001').stat().st_size / (1024 * 1024), 2))


In [ ]:
# 2. Job configuration
from pathlib import Path

# Dataset root tr?n Kaggle. S?a path n?y theo dataset b?n add b?ng n?t Add Input.
# V? d?: /kaggle/input/aic-video-part-01 ho?c /kaggle/input/aic2026-videos
DATASET_ROOT = Path('/kaggle/input/your-video-dataset')

# M?i Kaggle account nh?n kho?ng 3 folder trong 15 folder.
# ?i?n t?n folder ??ng nh? b?n trong DATASET_ROOT.
# N?u ?? r?ng [], notebook s? scan to?n b? DATASET_ROOT.
ASSIGNED_FOLDERS = [
    # 'L21',
    # 'L22',
    # 'L23',
]

PART_NAME = 'part_01'
SPLIT_NAME = 'Train'

SAMPLING_MODE = 'scene-uniform'
SAMPLE_INTERVAL_SECONDS = 2.0
MAX_FRAMES_PER_SCENE = 8

# WebP lossless ??p nh?ng ch?m. JPEG nhanh h?n nhi?u v? ?? t?t cho visual embedding.
IMAGE_FORMAT = 'jpg'  # 'webp' ho?c 'jpg'
JPEG_QUALITY = 95

# Ch?y th? N video ??u ?? ?o t?c ?? th?t. Set None ?? ch?y to?n b?.
MAX_VIDEOS_TO_PROCESS = None

# Kaggle x2 T4 support. N?u notebook ch? c? 1 GPU, code s? t? fallback v? single worker.
USE_DUAL_GPU_WORKERS = True
GPU_WORKER_IDS = ['0', '1']

VIDEO_EXTENSIONS = {'.mp4', '.mkv', '.avi', '.mov', '.webm', '.m4v', '.ts', '.wmv'}

OUTPUT_ROOT = Path('/kaggle/working') / f'{PART_NAME}_sbd_keyframes'
KEYFRAMES_ROOT = OUTPUT_ROOT / 'Keyframes' / SPLIT_NAME
MAP_KEYFRAMES_ROOT = OUTPUT_ROOT / 'map-keyframes'
METADATA_OUT = OUTPUT_ROOT / f'metadata_{PART_NAME}.json'
SUMMARY_OUT = OUTPUT_ROOT / f'video_summary_{PART_NAME}.csv'
ZIP_OUT = Path('/kaggle/working') / f'{PART_NAME}_sbd_keyframes.zip'

print('DATASET_ROOT:', DATASET_ROOT)
print('ASSIGNED_FOLDERS:', ASSIGNED_FOLDERS or '[all folders]')
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('Sampling:', SAMPLING_MODE, SAMPLE_INTERVAL_SECONDS, 'sec, max', MAX_FRAMES_PER_SCENE, 'frames/scene')
print('Dual GPU workers:', USE_DUAL_GPU_WORKERS, GPU_WORKER_IDS)
print('Image format:', IMAGE_FORMAT, 'JPEG quality:', JPEG_QUALITY)
print('Max videos to process:', MAX_VIDEOS_TO_PROCESS)


In [ ]:
# 3. Discover videos for this Kaggle job
from pathlib import Path
import os
import time

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f'DATASET_ROOT does not exist: {DATASET_ROOT}. '
        'Use Kaggle Add Input, then update DATASET_ROOT in the config cell.'
    )

print('Top-level entries under DATASET_ROOT:')
for child in sorted(DATASET_ROOT.iterdir())[:50]:
    print(' -', child.name, '[dir]' if child.is_dir() else '[file]')

search_roots = []
if ASSIGNED_FOLDERS:
    for folder_name in ASSIGNED_FOLDERS:
        folder = DATASET_ROOT / folder_name
        if not folder.exists():
            print(f'WARNING: assigned folder not found: {folder}')
            continue
        search_roots.append(folder)
else:
    print('WARNING: ASSIGNED_FOLDERS is empty, scanning the whole DATASET_ROOT. This can be slow if DATASET_ROOT is /kaggle/input.')
    search_roots = [DATASET_ROOT]

SKIP_DIR_NAMES = {
    '.git', '__MACOSX', 'transnetv2-weights', 'TransNetV2',
    'keyframes', 'Keyframes', 'map-keyframes', 'worker-artifacts',
}


def discover_videos_fast(root: Path):
    found = []
    started = time.time()
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [name for name in dirnames if name not in SKIP_DIR_NAMES and not name.startswith('.')]
        for filename in filenames:
            if Path(filename).suffix.lower() in VIDEO_EXTENSIONS:
                found.append(Path(dirpath) / filename)
    elapsed = time.time() - started
    return found, elapsed

videos = []
for root in search_roots:
    root_videos, elapsed = discover_videos_fast(root)
    print(f'{root}: found {len(root_videos)} videos in {elapsed:.2f}s')
    videos.extend(root_videos)

videos = sorted(set(videos))
print(f'Found {len(videos)} videos total')
for sample in videos[:10]:
    print('-', sample)
if len(videos) > 10:
    print('...')

if MAX_VIDEOS_TO_PROCESS is not None:
    videos = videos[:int(MAX_VIDEOS_TO_PROCESS)]
    print(f'Limited to first {len(videos)} videos for benchmark/test run')

if not videos:
    print('No videos found. Check DATASET_ROOT and ASSIGNED_FOLDERS in cell 2.')


In [ ]:
# 4. Extraction helpers
import csv
import json
import time
import traceback

import cv2
from PIL import Image
from tqdm.auto import tqdm
from transnetv2 import TransNetV2


def save_frame_image(frame_bgr, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(frame_rgb)
    fmt = IMAGE_FORMAT.lower().strip()
    if fmt in {'jpg', 'jpeg'}:
        image.save(destination, 'JPEG', quality=int(JPEG_QUALITY), optimize=False, progressive=False)
    elif fmt == 'webp':
        image.save(destination, 'WEBP', lossless=True, quality=100, method=6)
    else:
        raise ValueError(f'Unsupported IMAGE_FORMAT: {IMAGE_FORMAT}')
    return image.size


def frame_extension():
    fmt = IMAGE_FORMAT.lower().strip()
    return 'jpg' if fmt in {'jpg', 'jpeg'} else fmt


def frame_indices_for_scene(start_frame, end_frame, fps, sampling_mode, sample_interval_seconds, max_frames_per_scene):
    start_frame = int(start_frame)
    end_frame = int(end_frame)
    middle_frame_idx = int((start_frame + end_frame) // 2)

    if sampling_mode == 'middle':
        return [middle_frame_idx]

    frame_count = max(1, end_frame - start_frame + 1)
    interval_frames = max(1, int(round(float(fps) * float(sample_interval_seconds))))
    if frame_count <= interval_frames:
        return [middle_frame_idx]

    indices = list(range(start_frame, end_frame + 1, interval_frames))
    if middle_frame_idx not in indices:
        indices.append(middle_frame_idx)
    if end_frame not in indices:
        indices.append(end_frame)

    indices = sorted(set(indices))
    if max_frames_per_scene > 0 and len(indices) > max_frames_per_scene:
        if max_frames_per_scene == 1:
            return [middle_frame_idx]
        step = (len(indices) - 1) / float(max_frames_per_scene - 1)
        selected = [indices[round(i * step)] for i in range(max_frames_per_scene)]
        if middle_frame_idx not in selected:
            selected[len(selected) // 2] = middle_frame_idx
        indices = sorted(set(selected))

    return indices


def write_map_keyframes(video_id, video_metadata):
    MAP_KEYFRAMES_ROOT.mkdir(parents=True, exist_ok=True)
    csv_path = MAP_KEYFRAMES_ROOT / f'{video_id}.csv'
    with csv_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['n', 'pts_time', 'fps', 'frame_idx'])
        writer.writeheader()
        for item in sorted(video_metadata, key=lambda meta: meta['frame_index']):
            writer.writerow({
                'n': int(item['frame_index']) + 1,
                'pts_time': f"{float(item['timestamp']):.6f}",
                'fps': f"{float(item['fps']):.6f}",
                'frame_idx': int(item['global_frame_id']),
            })


def process_video(video_path: Path, model, global_faiss_id: int):
    video_id = video_path.stem
    video_output_dir = KEYFRAMES_ROOT / video_id
    video_output_dir.mkdir(parents=True, exist_ok=True)

    total_start = time.time()
    print(f'[{video_id}] phase 1/4: TransNetV2 predict scenes...', flush=True)
    predict_start = time.time()
    _, single_frame_predictions, _ = model.predict_video(str(video_path))
    scenes = model.predictions_to_scenes(single_frame_predictions)
    predict_elapsed = time.time() - predict_start
    print(f'[{video_id}] scenes: {len(scenes)} detected in {predict_elapsed:.2f}s', flush=True)

    print(f'[{video_id}] phase 2/4: open video and prepare sampling...', flush=True)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open video: {video_path}')

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 25.0)
    source_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    source_duration = source_frame_count / fps if fps else 0.0

    planned_samples = []
    for scene_index, scene in enumerate(scenes):
        start_frame, end_frame = scene
        for sample_index, frame_idx in enumerate(frame_indices_for_scene(
            start_frame,
            end_frame,
            fps=fps,
            sampling_mode=SAMPLING_MODE,
            sample_interval_seconds=SAMPLE_INTERVAL_SECONDS,
            max_frames_per_scene=MAX_FRAMES_PER_SCENE,
        )):
            planned_samples.append((scene_index, int(start_frame), int(end_frame), sample_index, int(frame_idx)))
    print(f'[{video_id}] planned keyframes: {len(planned_samples)} from {len(scenes)} scenes, duration {source_duration/60:.1f} min', flush=True)

    print(f'[{video_id}] phase 3/4: read frames and save {IMAGE_FORMAT}...', flush=True)
    save_start = time.time()
    video_metadata = []
    saved_frame_index = 0
    ext = frame_extension()
    progress_every = max(25, len(planned_samples) // 10) if planned_samples else 25

    for planned_index, (scene_index, start_frame, end_frame, sample_index, frame_idx) in enumerate(planned_samples, start=1):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame_bgr = cap.read()
        if not ok:
            print(f'WARNING: failed to read {video_id} frame {frame_idx}', flush=True)
            continue

        frame_name = f'keyframe_{saved_frame_index:04d}.{ext}'
        width, height = save_frame_image(frame_bgr, video_output_dir / frame_name)

        video_metadata.append({
            'faiss_id': global_faiss_id,
            'split': SPLIT_NAME,
            'video_id': video_id,
            'frame_name': frame_name,
            'frame_index': saved_frame_index,
            'global_frame_id': int(frame_idx),
            'timestamp': float(frame_idx) / fps,
            'fps': fps,
            'resolution': f'{width}x{height}',
            'scene_index': int(scene_index),
            'scene_start': int(start_frame),
            'scene_end': int(end_frame),
            'sample_index': int(sample_index),
            'samples_in_scene': 0,
            'sampling_mode': SAMPLING_MODE,
            'image_format': IMAGE_FORMAT,
            'source_video_path': str(video_path),
        })
        saved_frame_index += 1
        global_faiss_id += 1

        if planned_index % progress_every == 0 or planned_index == len(planned_samples):
            print(f'[{video_id}] saved {planned_index}/{len(planned_samples)} planned frames', flush=True)

    cap.release()
    save_elapsed = time.time() - save_start

    print(f'[{video_id}] phase 4/4: write map metadata...', flush=True)
    write_map_keyframes(video_id, video_metadata)

    summary = {
        'video_id': video_id,
        'source_path': str(video_path),
        'fps': fps,
        'source_frames': source_frame_count,
        'source_duration_seconds': source_duration,
        'scenes': len(scenes),
        'keyframes': len(video_metadata),
        'keyframes_per_second': len(video_metadata) / source_duration if source_duration else 0.0,
        'predict_elapsed_seconds': predict_elapsed,
        'save_elapsed_seconds': save_elapsed,
        'elapsed_seconds': time.time() - total_start,
        'status': 'ok',
        'error': '',
    }
    return global_faiss_id, video_metadata, summary


In [ ]:
# 5. Run extraction
import math
import subprocess
import os
import textwrap

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
KEYFRAMES_ROOT.mkdir(parents=True, exist_ok=True)
MAP_KEYFRAMES_ROOT.mkdir(parents=True, exist_ok=True)


def available_gpu_ids():
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=index', '--format=csv,noheader'],
            check=True,
            capture_output=True,
            text=True,
        )
        return [line.strip() for line in result.stdout.splitlines() if line.strip()]
    except Exception as exc:
        print('Could not query nvidia-smi, fallback to single worker:', exc)
        return []


def chunk_round_robin(items, chunks):
    buckets = [[] for _ in range(chunks)]
    for index, item in enumerate(items):
        buckets[index % chunks].append(str(item))
    return buckets


def write_worker_script(script_path: Path):
    worker_code = r'''
import csv
import json
import os
import sys
import time
import traceback
from pathlib import Path

import cv2
from PIL import Image
from tqdm.auto import tqdm

config_path = Path(sys.argv[1])
video_list_path = Path(sys.argv[2])
worker_index = int(sys.argv[3])

config = json.loads(config_path.read_text(encoding='utf-8'))
videos = [Path(item) for item in json.loads(video_list_path.read_text(encoding='utf-8'))]

sys.path.append(config['transnet_inference_path'])
from transnetv2 import TransNetV2

SPLIT_NAME = config['split_name']
SAMPLING_MODE = config['sampling_mode']
SAMPLE_INTERVAL_SECONDS = float(config['sample_interval_seconds'])
MAX_FRAMES_PER_SCENE = int(config['max_frames_per_scene'])
IMAGE_FORMAT = config.get('image_format', 'jpg')
JPEG_QUALITY = int(config.get('jpeg_quality', 95))
KEYFRAMES_ROOT = Path(config['keyframes_root'])
MAP_KEYFRAMES_ROOT = Path(config['map_keyframes_root'])
WORKER_OUT = Path(config['worker_out'])
WORKER_OUT.mkdir(parents=True, exist_ok=True)


def save_frame_image(frame_bgr, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(frame_rgb)
    fmt = IMAGE_FORMAT.lower().strip()
    if fmt in {'jpg', 'jpeg'}:
        image.save(destination, 'JPEG', quality=int(JPEG_QUALITY), optimize=False, progressive=False)
    elif fmt == 'webp':
        image.save(destination, 'WEBP', lossless=True, quality=100, method=6)
    else:
        raise ValueError(f'Unsupported IMAGE_FORMAT: {IMAGE_FORMAT}')
    return image.size


def frame_extension():
    fmt = IMAGE_FORMAT.lower().strip()
    return 'jpg' if fmt in {'jpg', 'jpeg'} else fmt


def frame_indices_for_scene(start_frame, end_frame, fps, sampling_mode, sample_interval_seconds, max_frames_per_scene):
    start_frame = int(start_frame)
    end_frame = int(end_frame)
    middle_frame_idx = int((start_frame + end_frame) // 2)

    if sampling_mode == 'middle':
        return [middle_frame_idx]

    frame_count = max(1, end_frame - start_frame + 1)
    interval_frames = max(1, int(round(float(fps) * float(sample_interval_seconds))))
    if frame_count <= interval_frames:
        return [middle_frame_idx]

    indices = list(range(start_frame, end_frame + 1, interval_frames))
    if middle_frame_idx not in indices:
        indices.append(middle_frame_idx)
    if end_frame not in indices:
        indices.append(end_frame)

    indices = sorted(set(indices))
    if max_frames_per_scene > 0 and len(indices) > max_frames_per_scene:
        if max_frames_per_scene == 1:
            return [middle_frame_idx]
        step = (len(indices) - 1) / float(max_frames_per_scene - 1)
        selected = [indices[round(i * step)] for i in range(max_frames_per_scene)]
        if middle_frame_idx not in selected:
            selected[len(selected) // 2] = middle_frame_idx
        indices = sorted(set(selected))

    return indices


def write_map_keyframes(video_id, video_metadata):
    MAP_KEYFRAMES_ROOT.mkdir(parents=True, exist_ok=True)
    csv_path = MAP_KEYFRAMES_ROOT / f'{video_id}.csv'
    with csv_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['n', 'pts_time', 'fps', 'frame_idx'])
        writer.writeheader()
        for item in sorted(video_metadata, key=lambda meta: meta['frame_index']):
            writer.writerow({
                'n': int(item['frame_index']) + 1,
                'pts_time': f"{float(item['timestamp']):.6f}",
                'fps': f"{float(item['fps']):.6f}",
                'frame_idx': int(item['global_frame_id']),
            })


def process_video(video_path: Path, model, local_id: int):
    video_id = video_path.stem
    video_output_dir = KEYFRAMES_ROOT / video_id
    video_output_dir.mkdir(parents=True, exist_ok=True)

    start_time = time.time()
    print(f'[{video_id}] worker {worker_index}: predict scenes...', flush=True)
    predict_start = time.time()
    _, single_frame_predictions, _ = model.predict_video(str(video_path))
    scenes = model.predictions_to_scenes(single_frame_predictions)
    predict_elapsed = time.time() - predict_start
    print(f'[{video_id}] worker {worker_index}: {len(scenes)} scenes in {predict_elapsed:.2f}s', flush=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open video: {video_path}')

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 25.0)
    source_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    source_duration = source_frame_count / fps if fps else 0.0

    planned_samples = []
    for scene_index, scene in enumerate(scenes):
        start_frame, end_frame = scene
        frame_indices = frame_indices_for_scene(
            start_frame,
            end_frame,
            fps=fps,
            sampling_mode=SAMPLING_MODE,
            sample_interval_seconds=SAMPLE_INTERVAL_SECONDS,
            max_frames_per_scene=MAX_FRAMES_PER_SCENE,
        )

        for sample_index, frame_idx in enumerate(frame_indices):
            planned_samples.append((scene_index, int(start_frame), int(end_frame), sample_index, int(frame_idx)))

    print(f'[{video_id}] worker {worker_index}: planned {len(planned_samples)} keyframes, saving {IMAGE_FORMAT}...', flush=True)
    save_start = time.time()
    video_metadata = []
    saved_frame_index = 0
    ext = frame_extension()
    progress_every = max(25, len(planned_samples) // 10) if planned_samples else 25

    for planned_index, (scene_index, start_frame, end_frame, sample_index, frame_idx) in enumerate(planned_samples, start=1):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
            ok, frame_bgr = cap.read()
            if not ok:
                print(f'WARNING: failed to read {video_id} frame {frame_idx}', flush=True)
                continue

            frame_name = f'keyframe_{saved_frame_index:04d}.{ext}'
            width, height = save_frame_image(frame_bgr, video_output_dir / frame_name)

            video_metadata.append({
                'split': SPLIT_NAME,
                'video_id': video_id,
                'frame_name': frame_name,
                'frame_index': saved_frame_index,
                'global_frame_id': int(frame_idx),
                'timestamp': float(frame_idx) / fps,
                'fps': fps,
                'resolution': f'{width}x{height}',
                'scene_index': int(scene_index),
                'scene_start': int(start_frame),
                'scene_end': int(end_frame),
                'sample_index': int(sample_index),
                'samples_in_scene': int(len(frame_indices)),
                'sampling_mode': SAMPLING_MODE,
                'source_video_path': str(video_path),
                'image_format': IMAGE_FORMAT,
                'worker_index': worker_index,
            })
            local_id += 1
            saved_frame_index += 1
            if planned_index % progress_every == 0 or planned_index == len(planned_samples):
                print(f'[{video_id}] worker {worker_index}: saved {planned_index}/{len(planned_samples)} planned frames', flush=True)

    save_elapsed = time.time() - save_start
    cap.release()
    write_map_keyframes(video_id, video_metadata)

    summary = {
        'video_id': video_id,
        'source_path': str(video_path),
        'fps': fps,
        'source_frames': source_frame_count,
        'source_duration_seconds': source_duration,
        'scenes': len(scenes),
        'keyframes': len(video_metadata),
        'keyframes_per_second': len(video_metadata) / source_duration if source_duration else 0.0,
        'predict_elapsed_seconds': predict_elapsed,
        'save_elapsed_seconds': save_elapsed,
        'elapsed_seconds': time.time() - start_time,
        'status': 'ok',
        'error': '',
        'worker_index': worker_index,
    }
    return local_id, video_metadata, summary

print(f'Worker {worker_index} starting on CUDA_VISIBLE_DEVICES={os.environ.get("CUDA_VISIBLE_DEVICES")} with {len(videos)} videos', flush=True)
model = TransNetV2()

local_id = 0
metadata = []
summaries = []
for video_path in tqdm(videos, desc=f'Worker {worker_index}'):
    try:
        local_id, video_metadata, summary = process_video(video_path, model, local_id)
        metadata.extend(video_metadata)
        summaries.append(summary)
        print(f"OK {summary['video_id']}: {summary['keyframes']} keyframes / {summary['scenes']} scenes", flush=True)
    except Exception as exc:
        traceback.print_exc()
        summaries.append({
            'video_id': video_path.stem,
            'source_path': str(video_path),
            'fps': 0,
            'source_frames': 0,
            'source_duration_seconds': 0,
            'scenes': 0,
            'keyframes': 0,
            'keyframes_per_second': 0,
            'elapsed_seconds': 0,
            'status': 'failed',
            'error': repr(exc),
            'worker_index': worker_index,
        })

metadata_path = WORKER_OUT / f'metadata_worker_{worker_index}.json'
summary_path = WORKER_OUT / f'summary_worker_{worker_index}.csv'
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

with summary_path.open('w', encoding='utf-8', newline='') as f:
    fieldnames = [
        'video_id', 'source_path', 'fps', 'source_frames', 'source_duration_seconds',
        'scenes', 'keyframes', 'keyframes_per_second', 'elapsed_seconds', 'status', 'error', 'worker_index',
    ]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(summaries)

print(f'Worker {worker_index} done. Metadata: {metadata_path}. Summary: {summary_path}', flush=True)
'''
    script_path.write_text(worker_code, encoding='utf-8')


all_metadata = {}
summaries = []

gpu_ids = available_gpu_ids()
worker_gpu_ids = [gpu for gpu in GPU_WORKER_IDS if gpu in gpu_ids]
use_parallel_workers = USE_DUAL_GPU_WORKERS and len(worker_gpu_ids) >= 2 and len(videos) >= 2

if use_parallel_workers:
    print('Running dual GPU workers:', worker_gpu_ids[:2])
    WORKER_OUT = OUTPUT_ROOT / 'worker-artifacts'
    WORKER_OUT.mkdir(parents=True, exist_ok=True)

    worker_script = Path('/kaggle/working/sbd_worker.py')
    write_worker_script(worker_script)

    config = {
        'transnet_inference_path': str(TRANSNET_DIR / 'inference'),
        'split_name': SPLIT_NAME,
        'sampling_mode': SAMPLING_MODE,
        'sample_interval_seconds': SAMPLE_INTERVAL_SECONDS,
        'max_frames_per_scene': MAX_FRAMES_PER_SCENE,
        'image_format': IMAGE_FORMAT,
        'jpeg_quality': JPEG_QUALITY,
        'keyframes_root': str(KEYFRAMES_ROOT),
        'map_keyframes_root': str(MAP_KEYFRAMES_ROOT),
        'worker_out': str(WORKER_OUT),
    }
    config_path = Path('/kaggle/working') / f'{PART_NAME}_worker_config.json'
    config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')

    buckets = chunk_round_robin(videos, 2)
    processes = []
    log_files = []
    for worker_index, bucket in enumerate(buckets):
        video_list_path = Path('/kaggle/working') / f'{PART_NAME}_worker_{worker_index}_videos.json'
        video_list_path.write_text(json.dumps(bucket, indent=2), encoding='utf-8')

        log_path = WORKER_OUT / f'worker_{worker_index}.log'
        log_file = log_path.open('w', encoding='utf-8')
        log_files.append(log_file)

        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = worker_gpu_ids[worker_index]
        env['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

        proc = subprocess.Popen(
            [sys.executable, str(worker_script), str(config_path), str(video_list_path), str(worker_index)],
            stdout=log_file,
            stderr=subprocess.STDOUT,
            env=env,
        )
        processes.append((worker_index, proc, log_path))
        print(f'Started worker {worker_index} on GPU {worker_gpu_ids[worker_index]} with {len(bucket)} videos. Log: {log_path}')

    def tail_progress_lines(log_path, max_lines=12):
        if not log_path.exists():
            return []
        text = log_path.read_text(encoding='utf-8', errors='replace')
        interesting = []
        for line in text.splitlines():
            if any(token in line for token in [
                'Worker ', 'predict scenes', 'scenes in', 'planned', 'saved ', 'OK ', 'WARNING:', 'Traceback', 'Error', 'Exception'
            ]):
                interesting.append(line)
        return interesting[-max_lines:]

    while any(proc.poll() is None for _, proc, _ in processes):
        time.sleep(60)
        print('--- parent heartbeat ---')
        for worker_index, proc, log_path in processes:
            status = 'running' if proc.poll() is None else f'exit {proc.returncode}'
            print(f'Worker {worker_index}: {status}. Log: {log_path}')
            for line in tail_progress_lines(log_path, max_lines=8):
                print(f'  {line}')

    for log_file in log_files:
        log_file.close()

    print('--- worker completion ---')
    for worker_index, proc, log_path in processes:
        print(f'Worker {worker_index}: exit {proc.returncode}. Log: {log_path}')
        for line in tail_progress_lines(log_path, max_lines=12):
            print(f'  {line}')

    failed = [(worker_index, proc.returncode, log_path) for worker_index, proc, log_path in processes if proc.returncode != 0]
    if failed:
        raise RuntimeError(f'One or more SBD workers failed: {failed}')

    merged_items = []
    for worker_index, _, _ in processes:
        worker_metadata_path = WORKER_OUT / f'metadata_worker_{worker_index}.json'
        worker_summary_path = WORKER_OUT / f'summary_worker_{worker_index}.csv'
        merged_items.extend(json.loads(worker_metadata_path.read_text(encoding='utf-8')))
        with worker_summary_path.open('r', encoding='utf-8', newline='') as f:
            summaries.extend(list(csv.DictReader(f)))

    for item in sorted(merged_items, key=lambda meta: (meta['video_id'], int(meta['frame_index']))):
        all_metadata[str(len(all_metadata))] = item

else:
    print('Running single worker in notebook process')
    print('Loading TransNetV2...')
    model = TransNetV2()
    print('Model ready')

    global_faiss_id = 0
    for video_path in tqdm(videos, desc='Videos'):
        try:
            global_faiss_id, video_metadata, summary = process_video(video_path, model, global_faiss_id)
            for item in video_metadata:
                faiss_id = item.pop('faiss_id')
                all_metadata[str(faiss_id)] = item
            summaries.append(summary)
            print(f"OK {summary['video_id']}: {summary['keyframes']} keyframes / {summary['scenes']} scenes")
        except Exception as exc:
            traceback.print_exc()
            summaries.append({
                'video_id': video_path.stem,
                'source_path': str(video_path),
                'fps': 0,
                'source_frames': 0,
                'source_duration_seconds': 0,
                'scenes': 0,
                'keyframes': 0,
                'keyframes_per_second': 0,
                'elapsed_seconds': 0,
                'status': 'failed',
                'error': repr(exc),
            })

with METADATA_OUT.open('w', encoding='utf-8') as f:
    json.dump(all_metadata, f, ensure_ascii=False, indent=2)

with SUMMARY_OUT.open('w', encoding='utf-8', newline='') as f:
    fieldnames = [
        'video_id', 'source_path', 'fps', 'source_frames', 'source_duration_seconds',
        'scenes', 'keyframes', 'keyframes_per_second', 'elapsed_seconds', 'status', 'error', 'worker_index',
    ]
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(summaries)

print('Total keyframes:', len(all_metadata))
print('Metadata:', METADATA_OUT)
print('Summary:', SUMMARY_OUT)
print('Map keyframes:', MAP_KEYFRAMES_ROOT)
print('Keyframes:', KEYFRAMES_ROOT)


In [ ]:
# 6. Quick sanity check + failure diagnostics
from collections import Counter
from pathlib import Path
import csv

status_counts = Counter(row['status'] for row in summaries)
print('Status:', dict(status_counts))

if summaries:
    total_duration = sum(float(row.get('source_duration_seconds') or 0) for row in summaries)
    total_keyframes = sum(int(float(row.get('keyframes') or 0)) for row in summaries)
    total_scenes = sum(int(float(row.get('scenes') or 0)) for row in summaries)
    print('Videos:', len(summaries))
    print('Total duration seconds:', round(total_duration, 2))
    print('Total scenes:', total_scenes)
    print('Total keyframes:', total_keyframes)
    print('Avg keyframes/sec:', round(total_keyframes / total_duration, 4) if total_duration else 0)

failed_rows = [row for row in summaries if row.get('status') != 'ok']
if failed_rows:
    print('\nFAILED SAMPLE ERRORS:')
    for row in failed_rows[:10]:
        print('-', row.get('video_id'), 'worker', row.get('worker_index', ''), ':', row.get('error', ''))

    worker_log_dir = OUTPUT_ROOT / 'worker-artifacts'
    if worker_log_dir.exists():
        print('\nWORKER LOG TAILS:')
        for log_path in sorted(worker_log_dir.glob('worker_*.log')):
            print(f'--- {log_path} ---')
            text = log_path.read_text(encoding='utf-8', errors='replace')
            lines = text.splitlines()
            for line in lines[-80:]:
                print(line)
else:
    print('\nNo failed videos detected.')

print('\nSample metadata items:')
for key in list(all_metadata.keys())[:3]:
    print(key, all_metadata[key])


In [ ]:
# 7. Zip artifacts for download / Kaggle output
import shutil

if ZIP_OUT.exists():
    ZIP_OUT.unlink()

archive_base = str(ZIP_OUT.with_suffix(''))
shutil.make_archive(archive_base, 'zip', root_dir=OUTPUT_ROOT)
print('Created:', ZIP_OUT)
print('Zip size GB:', round(ZIP_OUT.stat().st_size / (1024 ** 3), 3))


## Merge sau khi t?i v? m?y ch?nh

V?i 5 account Kaggle, t?i c?c file zip v? r?i gi?i n?n/gom nh? sau:

```text
src/data/Keyframes/Train/<video_id>/*.webp
src/dict/map-keyframes/<video_id>.csv
metadata_part_01.json
metadata_part_02.json
...
```

Kh?ng c?n gi? nguy?n `faiss_id` trong t?ng part. Khi merge metadata tr?n m?y ch?nh, n?n re-assign ID li?n t?c t? `0..N-1`, sau ?? encode l?i embeddings v? rebuild FAISS/Qdrant.